# Поиск дублей объявлений — Avito ML Cup 2025, триплеты


In [ ]:
import numpy as np

__all__ = ["mean_average_precision", "average_precision", "map_by_group"]


def average_precision(relevance: np.ndarray) -> float:
    # relevance — 0/1 в порядке убывания скора
    rel = np.asarray(relevance, dtype=np.float64)
    n_pos = rel.sum()
    if n_pos == 0:
        return 0.0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / n_pos)


def map_by_group(
    groups: np.ndarray,
    y_true: np.ndarray,
    scores: np.ndarray,
    skip_empty: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # AP по всем группам за одну сортировку — цикл по группам на миллионах пар не живёт
    groups = np.asarray(groups)
    y = np.asarray(y_true).astype(np.float64)
    s = np.asarray(scores, dtype=np.float64)
    if not (len(groups) == len(y) == len(s)):
        raise ValueError("groups, y_true и scores должны быть одной длины")
    if len(groups) == 0:
        empty = np.array([])
        return empty, empty, empty

    # первичный ключ — группа, вторичный — скор по убыванию; в lexsort главный ключ последний
    order = np.lexsort((-s, groups))
    g = groups[order]
    y = y[order]

    n = len(y)
    starts = np.flatnonzero(np.concatenate(([True], g[1:] != g[:-1])))
    sizes = np.diff(np.concatenate((starts, [n])))
    row_start = np.repeat(starts, sizes)  # начало своей группы для каждой строки
    rank = np.arange(n) - row_start + 1  # позиция внутри группы, с 1

    # cum_excl[i] = сумма y[:i], значит попаданий до i включительно = cum_excl[i+1] - cum_excl[start]
    cum_excl = np.concatenate(([0.0], np.cumsum(y)))
    hits = cum_excl[1:] - cum_excl[row_start]

    contrib = hits / rank * y
    ap_sum = np.add.reduceat(contrib, starts)
    n_pos = np.add.reduceat(y, starts)
    ap = np.divide(ap_sum, n_pos, out=np.zeros_like(ap_sum), where=n_pos > 0)

    group_ids = g[starts]
    if skip_empty:
        # базы без единого дубля не в счёт, для них AP не определён
        keep = n_pos > 0
        return group_ids[keep], ap[keep], n_pos[keep]
    return group_ids, ap, n_pos


def mean_average_precision(
    groups: np.ndarray,
    y_true: np.ndarray,
    scores: np.ndarray,
    skip_empty: bool = True,
) -> float:
    # среднее AP по базам. на сплошных ties зависит от порядка строк, для константы бессмысленна
    _, ap, _ = map_by_group(groups, y_true, scores, skip_empty=skip_empty)
    return float(ap.mean()) if len(ap) else 0.0

## нормализация текста и починка гомоглифов

In [ ]:
"""тексты обфусцированы: часть кириллицы подменена латинскими двойниками.

fold_confusables сводит обе азбуки к одной — когда сравниваю строки между собой.
restore_homoglyphs чинит слово в сторону его алфавита — перед подачей в модель.
"""


import re
import unicodedata
from typing import Iterable, Sequence

__all__ = [
    "fold_confusables",
    "restore_homoglyphs",
    "normalize",
    "tokenize",
    "jaccard",
    "char_ngrams",
    "normalize_vocabulary",
    "CONFUSABLES",
    "FOLD_PAIRS",
]

# пары «латинская — кириллическая», неразличимые в типичном шрифте. заглавные и строчные
# отдельно: К и K неразличимы, а к и k — вполне
CONFUSABLES: tuple[tuple[str, str], ...] = (
    ("a", "а"), ("c", "с"), ("e", "е"), ("o", "о"), ("p", "р"), ("x", "х"), ("y", "у"),
    ("A", "А"), ("B", "В"), ("C", "С"), ("E", "Е"), ("H", "Н"), ("K", "К"), ("M", "М"),
    ("O", "О"), ("P", "Р"), ("T", "Т"), ("X", "Х"), ("Y", "У"),
)

FOLD_PAIRS: tuple[tuple[str, str], ...] = (
    ("а", "a"), ("с", "c"), ("е", "e"), ("о", "o"), ("р", "p"), ("х", "x"), ("у", "y"),
    ("к", "k"), ("м", "m"), ("н", "h"), ("в", "b"), ("т", "t"),
)

_LATIN_AMBIGUOUS = {lat for lat, _ in CONFUSABLES}
_CYRILLIC_AMBIGUOUS = {cyr for _, cyr in CONFUSABLES}

_TO_CYRILLIC = str.maketrans({lat: cyr for lat, cyr in CONFUSABLES})
_TO_LATIN = str.maketrans({cyr: lat for lat, cyr in CONFUSABLES})
_FOLD = str.maketrans({cyr: lat for cyr, lat in FOLD_PAIRS})

_LATIN_RE = re.compile(r"[A-Za-z]")
_CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")
_WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)
_PUNCT_RE = re.compile(r"[^\w\s]|_", re.UNICODE)
_SPACE_RE = re.compile(r"\s+")


def fold_confusables(text: str) -> str:
    # ждёт нижний регистр. свёртка разрушающая, но починенных совпадений на порядки больше
    if not text:
        return ""
    return text.translate(_FOLD)


def _restore_token(token: str) -> str:
    # голосуют только однозначные буквы, у которых двойника нет
    has_real_cyrillic = False
    has_real_latin = False
    for ch in token:
        if ch not in _CYRILLIC_AMBIGUOUS and _CYRILLIC_RE.match(ch):
            has_real_cyrillic = True
        elif ch not in _LATIN_AMBIGUOUS and _LATIN_RE.match(ch):
            has_real_latin = True
    if has_real_cyrillic and not has_real_latin:
        return token.translate(_TO_CYRILLIC)
    if has_real_latin and not has_real_cyrillic:
        return token.translate(_TO_LATIN)
    # либо осмысленная смесь (iPhone), либо сплошные двойники (ecco, сор) — угадывать нечего
    return token


def restore_homoglyphs(text: str) -> str:
    # куpткa зимняя ecco -> куртка зимняя ecco, бренд остаётся латинским
    if not text:
        return ""
    return _WORD_RE.sub(lambda m: _restore_token(m.group(0)), text)


def normalize(text: str, *, fold: bool = True) -> str:
    # nfkc, нижний регистр, пунктуация в пробелы, пробелы схлопнуты; свёртка последней
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    if fold:
        text = fold_confusables(text)
    text = _PUNCT_RE.sub(" ", text)
    return _SPACE_RE.sub(" ", text).strip()


def tokenize(text: str, *, fold: bool = True) -> list[str]:
    # числа оставляю намеренно: в объявлениях они несут модель, размер и объём памяти —
    # самое различающее в парах вроде iphone 11 256 гб против iphone 11 128 гб
    return normalize(text, fold=fold).split()


def jaccard(a: Iterable[str], b: Iterable[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    union = len(sa | sb)
    return len(sa & sb) / union if union else 0.0


def char_ngrams(text: str, n: int = 3) -> set[str]:
    # устойчивы к опечаткам и другому порядку слов, значит ловят переписанные описания
    # там, где пословный жаккар уже проваливается
    s = normalize(text)
    if len(s) < n:
        return {s} if s else set()
    return {s[i : i + n] for i in range(len(s) - n + 1)}


def normalize_vocabulary(texts: Sequence[str], func) -> dict[str, str]:
    # уникальных токенов в корпусе на два порядка меньше, чем вхождений. считать по
    # словарю, а не по каждому вхождению — это секунды против часов на миллионах строк
    vocab: dict[str, str] = {}
    for text in texts:
        if not text:
            continue
        for token in _WORD_RE.findall(text):
            if token not in vocab:
                vocab[token] = func(token)
    return vocab

## парные признаки

In [ ]:
"""признаки описывают пару, а не объявление: модель видит только сходство base и cand."""


import json
from typing import Iterable

import numpy as np
import pandas as pd


__all__ = [
    "build_features",
    "TEXT_PAIRS",
    "add_embedding_features",
    "embedding_pair_features",
    "side_numeric_features",
]

# поле и размер символьной n-граммы. у заголовков n=3, они короткие и длиннее просто
# не наберётся; у описаний n=4 — текста хватает, а случайных совпадений меньше
TEXT_PAIRS: tuple[tuple[str, int], ...] = (("title", 3), ("description", 4))

MAX_DESCRIPTION_CHARS = 512


def _safe_str(series: pd.Series) -> np.ndarray:
    return series.fillna("").astype(str).to_numpy()


def _containment(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


def _lcp_ratio(a: str, b: str) -> float:
    # общий префикс — ловит iphone 11 128 гб против iphone 11 256 гб
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i / max(len(a), len(b))


def _ngrams(text: str, n: int) -> set[str]:
    if len(text) < n:
        return {text} if text else set()
    return {text[i : i + n] for i in range(len(text) - n + 1)}


class _BoundedCache:
    """пары одной базы идут подряд, поэтому кэш разобранных текстов окупается.
    но только ограниченный: без потолка он разросся до 12.6 гб на одном файле.
    """

    __slots__ = ("_prepare", "_maxsize", "_data")

    def __init__(self, prepare, maxsize: int = 4096) -> None:
        self._prepare = prepare
        self._maxsize = maxsize
        self._data: dict = {}

    def get(self, raw: str):
        hit = self._data.get(raw)
        if hit is None:
            hit = self._prepare(raw)
            if len(self._data) >= self._maxsize:
                self._data.clear()
            self._data[raw] = hit
        return hit


def _text_similarity_block(
    base_raw: np.ndarray, cand_raw: np.ndarray, field: str, ngram: int, max_chars: int | None
) -> dict[str, np.ndarray]:
    n = len(base_raw)
    out = {
        f"{field}_exact": np.zeros(n, dtype=np.float32),
        f"{field}_jaccard_tok": np.zeros(n, dtype=np.float32),
        f"{field}_containment_tok": np.zeros(n, dtype=np.float32),
        f"{field}_jaccard_ngram": np.zeros(n, dtype=np.float32),
        f"{field}_digits_jaccard": np.zeros(n, dtype=np.float32),
        f"{field}_lcp_ratio": np.zeros(n, dtype=np.float32),
        f"{field}_len_ratio": np.zeros(n, dtype=np.float32),
        f"{field}_len_diff": np.zeros(n, dtype=np.float32),
        f"{field}_both_empty": np.zeros(n, dtype=np.float32),
    }
    def prepare(raw: str):
        norm = normalize(raw[:max_chars] if max_chars else raw)
        toks = norm.split()
        return norm, set(toks), _ngrams(norm, ngram), {t for t in toks if t.isdigit()}

    cache = _BoundedCache(prepare)

    for i in range(n):
        b_norm, b_tok, b_ng, b_dig = cache.get(base_raw[i])
        c_norm, c_tok, c_ng, c_dig = cache.get(cand_raw[i])

        if not b_norm and not c_norm:
            out[f"{field}_both_empty"][i] = 1.0
            continue

        out[f"{field}_exact"][i] = float(b_norm == c_norm and bool(b_norm))
        out[f"{field}_jaccard_tok"][i] = jaccard(b_tok, c_tok)
        out[f"{field}_containment_tok"][i] = _containment(b_tok, c_tok)
        out[f"{field}_jaccard_ngram"][i] = jaccard(b_ng, c_ng)
        if b_dig or c_dig:
            out[f"{field}_digits_jaccard"][i] = jaccard(b_dig, c_dig)
        else:
            out[f"{field}_digits_jaccard"][i] = -1.0  # чисел нет ни там, ни там
        out[f"{field}_lcp_ratio"][i] = _lcp_ratio(b_norm, c_norm)

        lb, lc = len(b_norm), len(c_norm)
        out[f"{field}_len_ratio"][i] = min(lb, lc) / max(lb, lc) if max(lb, lc) else 0.0
        out[f"{field}_len_diff"][i] = abs(lb - lc)

    return out


def _parse_params(raw: str) -> dict:
    if not raw:
        return {}
    try:
        parsed = json.loads(raw)
    except (ValueError, TypeError):
        return {}
    return parsed if isinstance(parsed, dict) else {}


def _json_params_block(base_raw: np.ndarray, cand_raw: np.ndarray) -> dict[str, np.ndarray]:
    n = len(base_raw)
    out = {
        "params_n_common_keys": np.zeros(n, dtype=np.float32),
        "params_key_jaccard": np.zeros(n, dtype=np.float32),
        "params_value_match_ratio": np.zeros(n, dtype=np.float32),
        "params_n_base": np.zeros(n, dtype=np.float32),
        "params_n_cand": np.zeros(n, dtype=np.float32),
    }
    cache = _BoundedCache(_parse_params)

    for i in range(n):
        b, c = cache.get(base_raw[i]), cache.get(cand_raw[i])
        out["params_n_base"][i] = len(b)
        out["params_n_cand"][i] = len(c)
        if not b or not c:
            out["params_value_match_ratio"][i] = -1.0
            continue
        bk, ck = set(b), set(c)
        common = bk & ck
        out["params_n_common_keys"][i] = len(common)
        out["params_key_jaccard"][i] = len(common) / len(bk | ck)
        if common:
            equal = sum(1 for k in common if b[k] == c[k])
            out["params_value_match_ratio"][i] = equal / len(common)
        else:
            out["params_value_match_ratio"][i] = -1.0

    return out


def _clean_price(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64)
    return np.where(values > 0, values, np.nan)


def side_numeric_features(df: pd.DataFrame, prefix: str) -> np.ndarray:
    price = _clean_price(df[f"{prefix}_price"])
    log_price = np.log1p(np.nan_to_num(price, nan=0.0))
    price_known = np.isfinite(price).astype(np.float64)

    images = pd.to_numeric(df[f"{prefix}_count_images"], errors="coerce").fillna(0).to_numpy()
    title_len = df[f"{prefix}_title"].fillna("").str.len().to_numpy()
    desc_len = df[f"{prefix}_description"].fillna("").str.len().to_numpy()

    out = np.column_stack([
        log_price, price_known, images, np.log1p(title_len), np.log1p(desc_len)
    ]).astype(np.float32)
    if not np.isfinite(out).all():
        raise ValueError(f"в признаках стороны {prefix} остались NaN или бесконечности")
    return out


def _numeric_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # цена, число фотографий и готовые гео-флаги
    out: dict[str, np.ndarray] = {}

    bp = _clean_price(df["base_price"])
    cp = _clean_price(df["cand_price"])
    lo, hi = np.minimum(bp, cp), np.maximum(bp, cp)
    with np.errstate(divide="ignore", invalid="ignore"):
        out["price_ratio"] = np.where(hi > 0, lo / hi, np.nan)
        out["price_log_diff"] = np.abs(np.log1p(bp) - np.log1p(cp))
    out["price_abs_diff"] = np.abs(bp - cp)
    out["price_min"] = lo
    out["price_max"] = hi
    out["price_missing"] = (np.isnan(bp) | np.isnan(cp)).astype(np.float32)

    bi = pd.to_numeric(df["base_count_images"], errors="coerce").fillna(0).to_numpy()
    ci = pd.to_numeric(df["cand_count_images"], errors="coerce").fillna(0).to_numpy()
    out["images_abs_diff"] = np.abs(bi - ci)
    out["images_min"] = np.minimum(bi, ci)
    out["images_max"] = np.maximum(bi, ci)
    out["images_equal"] = (bi == ci).astype(np.float32)

    for col in ("is_same_location", "is_same_region"):
        if col in df.columns:
            out[col] = df[col].fillna(False).astype(np.float32).to_numpy()

    return out


def _categorical_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # разные категории почти исключают дубль, то есть это работает фильтром, а не слабым сигналом
    out: dict[str, np.ndarray] = {}
    for field in ("category_name", "subcategory_name", "param1", "param2"):
        b_col, c_col = f"base_{field}", f"cand_{field}"
        if b_col not in df.columns or c_col not in df.columns:
            continue
        b, c = _safe_str(df[b_col]), _safe_str(df[c_col])
        out[f"same_{field}"] = (b == c).astype(np.float32)
        out[f"{field}_missing"] = ((b == "") | (c == "")).astype(np.float32)
    return out


def _title_image_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    out: dict[str, np.ndarray] = {}
    if "base_title_image" not in df.columns:
        return out
    b = _safe_str(df["base_title_image"])
    c = _safe_str(df["cand_title_image"])
    out["title_image_both_present"] = ((b != "") & (c != "")).astype(np.float32)
    out["title_image_equal"] = (b == c).astype(np.float32)
    return out


def _cross_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # перепост часто копирует заголовок в описание, значит такое вложение ловит дубли,
    # у которых сами заголовки переписаны и напрямую не совпадают
    n = len(df)
    titles = {p: _safe_str(df[f"{p}_title"]) for p in ("base", "cand")}
    descriptions = {p: _safe_str(df[f"{p}_description"]) for p in ("base", "cand")}

    title_cache = _BoundedCache(lambda raw: set(tokenize(raw)))
    desc_cache = _BoundedCache(lambda raw: set(tokenize(raw[:MAX_DESCRIPTION_CHARS])))

    out = {
        "cross_base_title_in_cand_desc": np.zeros(n, dtype=np.float32),
        "cross_cand_title_in_base_desc": np.zeros(n, dtype=np.float32),
    }
    for i in range(n):
        out["cross_base_title_in_cand_desc"][i] = _containment(
            title_cache.get(titles["base"][i]), desc_cache.get(descriptions["cand"][i])
        )
        out["cross_cand_title_in_base_desc"][i] = _containment(
            title_cache.get(titles["cand"][i]), desc_cache.get(descriptions["base"][i])
        )
    return out


def build_features(df: pd.DataFrame, *, verbose: bool = False) -> pd.DataFrame:
    blocks: dict[str, np.ndarray] = {}

    for field, ngram in TEXT_PAIRS:
        if verbose:
            print(f"  признаки по полю {field}...", flush=True)
        max_chars = MAX_DESCRIPTION_CHARS if field == "description" else None
        blocks.update(
            _text_similarity_block(
                _safe_str(df[f"base_{field}"]),
                _safe_str(df[f"cand_{field}"]),
                field,
                ngram,
                max_chars,
            )
        )

    if "base_json_params" in df.columns:
        if verbose:
            print("  признаки по json_params...", flush=True)
        blocks.update(
            _json_params_block(_safe_str(df["base_json_params"]), _safe_str(df["cand_json_params"]))
        )

    if verbose:
        print("  числовые и категориальные...", flush=True)
    blocks.update(_numeric_block(df))
    blocks.update(_categorical_block(df))
    blocks.update(_title_image_block(df))

    if verbose:
        print("  перекрёстные заголовок/описание...", flush=True)
    blocks.update(_cross_block(df))

    out = pd.DataFrame(blocks, index=df.index)
    return out.astype(np.float32)


def add_embedding_features(
    features: pd.DataFrame,
    base_emb: np.ndarray,
    cand_emb: np.ndarray,
    prefix: str = "emb",
) -> pd.DataFrame:
    b = np.ascontiguousarray(base_emb, dtype=np.float32)
    c = np.ascontiguousarray(cand_emb, dtype=np.float32)
    bn = b / np.maximum(np.linalg.norm(b, axis=1, keepdims=True), 1e-8)
    cn = c / np.maximum(np.linalg.norm(c, axis=1, keepdims=True), 1e-8)

    diff = np.abs(bn - cn)
    features = features.copy()
    features[f"{prefix}_cosine"] = (bn * cn).sum(axis=1).astype(np.float32)
    features[f"{prefix}_l2"] = np.linalg.norm(bn - cn, axis=1).astype(np.float32)
    features[f"{prefix}_diff_mean"] = diff.mean(axis=1).astype(np.float32)
    features[f"{prefix}_diff_max"] = diff.max(axis=1).astype(np.float32)
    features[f"{prefix}_diff_std"] = diff.std(axis=1).astype(np.float32)
    return features


def embedding_pair_features(
    embeddings: np.ndarray,
    base_idx: np.ndarray,
    cand_idx: np.ndarray,
    prefix: str = "emb",
    chunk: int = 200_000,
    index: pd.Index | None = None,
) -> pd.DataFrame:
    n = len(base_idx)
    cols = ["cosine", "l2", "diff_mean", "diff_max", "diff_std"]
    out = {f"{prefix}_{c}": np.empty(n, dtype=np.float32) for c in cols}

    def take(idx: np.ndarray) -> np.ndarray:
        rows = embeddings[idx].astype(np.float32, copy=False)
        norms = np.maximum(np.linalg.norm(rows, axis=1, keepdims=True), 1e-8)
        return rows / norms

    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        b = take(base_idx[start:end])
        c = take(cand_idx[start:end])
        diff = np.abs(b - c)
        out[f"{prefix}_cosine"][start:end] = (b * c).sum(axis=1)
        out[f"{prefix}_l2"][start:end] = np.linalg.norm(b - c, axis=1)
        out[f"{prefix}_diff_mean"][start:end] = diff.mean(axis=1)
        out[f"{prefix}_diff_max"][start:end] = diff.max(axis=1)
        out[f"{prefix}_diff_std"][start:end] = diff.std(axis=1)

    return pd.DataFrame(out, index=index)

In [ ]:
"""одно объявление стоит и как base, и как cand, поэтому случайное разбиение по строкам
растаскивает его между train и valid и задирает оценку. режу по group_id, кластером целиком.
"""


import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

__all__ = ["group_folds", "leakage_report"]


def group_folds(
    groups: np.ndarray, n_splits: int = 5, seed: int = 42
) -> list[tuple[np.ndarray, np.ndarray]]:
    groups = np.asarray(groups)
    # GroupKFold раскладывает группы по размеру и не перемешивает, то есть сид на него
    # не влияет — перемешиваю сам, переименовав группы случайной перестановкой
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    shuffled = rng.permutation(len(uniq))
    remap = dict(zip(uniq, shuffled))
    permuted = np.array([remap[g] for g in groups])

    splitter = GroupKFold(n_splits=n_splits)
    dummy = np.zeros(len(groups))
    return list(splitter.split(dummy, groups=permuted))


def leakage_report(
    df: pd.DataFrame,
    train_idx: np.ndarray,
    valid_idx: np.ndarray,
    id_cols: tuple[str, ...] = ("base_item_id", "cand_item_id"),
) -> dict[str, float]:
    # доля объявлений валидации, которые уже были в обучении. по group_id должна быть нулевой
    train_ids: set = set()
    valid_ids: set = set()
    for col in id_cols:
        train_ids |= set(df.iloc[train_idx][col].dropna())
        valid_ids |= set(df.iloc[valid_idx][col].dropna())
    if not valid_ids:
        return {"overlap_ratio": 0.0, "n_valid_items": 0, "n_leaked_items": 0}
    leaked = valid_ids & train_ids
    return {
        "overlap_ratio": len(leaked) / len(valid_ids),
        "n_valid_items": len(valid_ids),
        "n_leaked_items": len(leaked),
    }

In [ ]:
__all__ = ["pick_device", "reset_device_cache"]

_CACHED: str | None = None


def pick_device(verbose: bool = True) -> str:
    global _CACHED
    if _CACHED is not None:
        return _CACHED

    import torch

    if not torch.cuda.is_available():
        _CACHED = "cpu"
        return _CACHED

    try:
        probe = torch.ones(8, 8, device="cuda")
        (probe @ probe).sum().item()
        _CACHED = "cuda"
        if verbose:
            print(f"устройство: cuda ({torch.cuda.get_device_name(0)})", flush=True)
    except Exception as exc:  # noqa: BLE001
        name = "неизвестна"
        try:
            name = torch.cuda.get_device_name(0)
        except Exception:  # noqa: BLE001
            pass
        if verbose:
            print(f"видеокарта {name} непригодна ({exc}); считаю на CPU", flush=True)
        _CACHED = "cpu"

    return _CACHED


def reset_device_cache() -> None:
    global _CACHED
    _CACHED = None

## embeddings

In [ ]:
"""эмбеддинги rubert-tiny2."""


import numpy as np


__all__ = ["TextIndex", "encode_texts", "DEFAULT_MODEL"]

DEFAULT_MODEL = "cointegrated/rubert-tiny2"


class TextIndex:
    def __init__(self) -> None:
        self._ids: dict[str, int] = {}
        self.texts: list[str] = []

    def add(self, text: str) -> int:
        idx = self._ids.get(text)
        if idx is None:
            idx = len(self.texts)
            self._ids[text] = idx
            self.texts.append(text)
        return idx

    def add_many(self, texts) -> np.ndarray:
        return np.fromiter((self.add(t) for t in texts), dtype=np.int64, count=len(texts))

    def __len__(self) -> int:
        return len(self.texts)


def build_bert_text(title: str, description: str, desc_chars: int = 128) -> str:
    title = restore_homoglyphs(title or "")
    desc = restore_homoglyphs((description or "")[:desc_chars])
    return f"{title}. {desc}".strip()


def encode_texts(
    texts: list[str],
    model_name: str = DEFAULT_MODEL,
    batch_size: int = 512,
    max_length: int = 64,
    device: str | None = None,
    verbose: bool = True,
) -> np.ndarray:
    import torch
    from transformers import AutoModel, AutoTokenizer

    if device is None:
        device = pick_device(verbose=verbose)
    if verbose:
        print(f"кодирую {len(texts)} уникальных текстов на {device}", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    # float16: уникальных текстов миллионы, в float32 таблица занимает больше, чем все
    # признаки вместе. точность не нужна — дальше из векторов только косинус и статистики
    dim = model.config.hidden_size
    out = np.empty((len(texts), dim), dtype=np.float16)

    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            out[start : start + len(batch)] = pooled.float().cpu().numpy().astype(np.float16)
            if verbose and start and start % (batch_size * 200) == 0:
                print(f"  {start}/{len(texts)}", flush=True)

    return out

## two-tower contrastive model

In [ ]:
"""косинус сырых эмбеддингов бустингу почти ничего не даёт: модель не дообучается и меряет
смысловую близость, а два зимних пуховика близки по смыслу, но не дубли. мне нужна близость
в смысле дубля, значит её надо выучивать.

замороженный энкодер даёт представление, поверх учу проекцию — она стягивает дубли и разводит
недубли. обе стороны идут через один энкодер независимо и встречаются только на расстоянии
между проекциями, то есть модель не может подсмотреть в кандидата, кодируя базу.
"""


import numpy as np


__all__ = ["TwoTowerContrastive", "contrastive_loss", "build_side_matrix", "SideView"]


class SideView:
    def __init__(self, embeddings: np.ndarray, idx: np.ndarray, numeric: np.ndarray) -> None:
        if len(idx) != len(numeric):
            raise ValueError("длина индексов и числовых признаков не совпадает")
        self.embeddings = embeddings
        self.idx = np.asarray(idx)
        self.numeric = np.asarray(numeric, dtype=np.float32)
        self.shape = (len(self.idx), embeddings.shape[1] + self.numeric.shape[1])

    def __len__(self) -> int:
        return self.shape[0]

    def take(self, rows: np.ndarray) -> np.ndarray:
        emb = self.embeddings[self.idx[rows]].astype(np.float32, copy=False)
        return np.hstack([emb, self.numeric[rows]])

    def subset(self, rows: np.ndarray) -> "SideView":
        # вид на подмножество — нарезка по фолдам без копирования
        return SideView(self.embeddings, self.idx[rows], self.numeric[rows])


class _DenseView:
    # тот же интерфейс поверх обычного массива, чтобы модель не различала случаи
    def __init__(self, array: np.ndarray) -> None:
        self.array = np.asarray(array, dtype=np.float32)
        self.shape = self.array.shape

    def __len__(self) -> int:
        return len(self.array)

    def take(self, rows: np.ndarray) -> np.ndarray:
        return self.array[rows]

    def subset(self, rows: np.ndarray) -> "_DenseView":
        return _DenseView(self.array[rows])


def _as_view(x):
    return x if hasattr(x, "take") and hasattr(x, "subset") else _DenseView(x)


def _torch():
    import torch

    return torch


def contrastive_loss(distance, label, margin: float = 1.0):
    # дубли штрафуются за расстояние, недубли — за то, что подошли ближе зазора. недубли
    # дальше margin вклада не дают, иначе обучение уходит в раздувание расстояний
    torch = _torch()
    positive = label * distance.pow(2)
    negative = (1.0 - label) * torch.clamp(margin - distance, min=0.0).pow(2)
    return (positive + negative).mean()


def _build_module(input_dim: int, hidden: int, output: int, dropout: float):
    torch = _torch()
    nn = torch.nn

    class TwoTower(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden),
                nn.BatchNorm1d(hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, output),
            )

        def encode(self, x):
            z = self.encoder(x)
            # l2: без неё расстояние можно уменьшать, просто сжимая представление,
            # и зазор перестаёт что-либо значить
            return z / z.norm(dim=1, keepdim=True).clamp(min=1e-8)

        def forward(self, x_base, x_cand):
            z_base = self.encode(x_base)
            z_cand = self.encode(x_cand)
            delta = z_base - z_cand
            return torch.sqrt((delta * delta).sum(dim=1) + 1e-12)

    return TwoTower()


class TwoTowerContrastive:
    def __init__(
        self,
        input_dim: int,
        hidden: int = 256,
        output: int = 128,
        margin: float = 1.0,
        dropout: float = 0.1,
        lr: float = 1e-3,
        batch_size: int = 4096,
        epochs: int = 5,
        seed: int = 42,
        device: str | None = None,
        verbose: bool = True,
    ) -> None:
        self.input_dim = input_dim
        self.hidden = hidden
        self.output = output
        self.margin = margin
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.seed = seed
        self.verbose = verbose
        self.device = device or pick_device(verbose=verbose)
        self.model = None

    def fit(self, x_base, x_cand, y: np.ndarray) -> "TwoTowerContrastive":
        torch = _torch()
        torch.manual_seed(self.seed)

        base_view, cand_view = _as_view(x_base), _as_view(x_cand)
        self.model = _build_module(self.input_dim, self.hidden, self.output, self.dropout)
        self.model = self.model.to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        y = np.asarray(y, dtype=np.float32)
        n = len(y)
        # дублей ~5%: без выравнивания батч почти целиком из недублей, градиент от
        # положительной части тонет в шуме и модель сходится к «всё далеко»
        positive_idx = np.flatnonzero(y == 1)
        negative_idx = np.flatnonzero(y == 0)
        if len(positive_idx) == 0 or len(negative_idx) == 0:
            raise ValueError("в обучающей выборке должны быть оба класса")
        rng = np.random.default_rng(self.seed)
        half = max(self.batch_size // 2, 1)
        steps = max(n // self.batch_size, 1)

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            total = 0.0
            for _ in range(steps):
                rows = np.concatenate([
                    rng.choice(positive_idx, size=half, replace=len(positive_idx) < half),
                    rng.choice(negative_idx, size=half, replace=len(negative_idx) < half),
                ])
                xb = torch.from_numpy(base_view.take(rows)).to(self.device)
                xc = torch.from_numpy(cand_view.take(rows)).to(self.device)
                yt = torch.from_numpy(y[rows]).to(self.device)

                optimizer.zero_grad()
                loss = contrastive_loss(self.model(xb, xc), yt, self.margin)
                loss.backward()
                optimizer.step()
                total += loss.item()
            if self.verbose:
                print(f"  эпоха {epoch}: loss {total / steps:.4f}", flush=True)
        return self

    def predict(self, x_base, x_cand, batch_size: int = 16384) -> np.ndarray:
        # 1/(1+d), а не -d: так значения лежат в (0, 1] и годятся как признак наравне с остальными
        torch = _torch()
        if self.model is None:
            raise RuntimeError("модель не обучена")
        base_view, cand_view = _as_view(x_base), _as_view(x_cand)
        self.model.eval()
        n = len(base_view)
        out = np.empty(n, dtype=np.float32)
        with torch.inference_mode():
            for start in range(0, n, batch_size):
                rows = np.arange(start, min(start + batch_size, n))
                b = torch.from_numpy(base_view.take(rows)).to(self.device)
                c = torch.from_numpy(cand_view.take(rows)).to(self.device)
                distance = self.model(b, c).cpu().numpy()
                out[rows] = 1.0 / (1.0 + distance)
        return out


def build_side_matrix(
    embeddings: np.ndarray, idx: np.ndarray, numeric: np.ndarray
) -> np.ndarray:
    # вход одной башни: эмбеддинг плюс числовые атрибуты. стандартизует вызывающий код
    emb = embeddings[idx].astype(np.float32, copy=False)
    return np.hstack([emb, numeric.astype(np.float32, copy=False)])

## triplet tower with hard negative mining

In [ ]:
"""anchor и positive беру из дубля, негатив домываю от узкого бакета к широкому: свои же
кандидаты той же базы, потом param2, param1, подкатегория, категория. случайный негатив
отделяется тривиально, и башня упирается в потолок.
"""


import numpy as np


__all__ = ["SideRefView", "mine_triplets", "TripletTower", "DEFAULT_MARGIN", "LEVELS"]

DEFAULT_MARGIN = 0.3

# от узкого бакета к широкому
LEVELS = ("та же база", "param2", "param1", "подкатегория", "категория", "случайный")


class SideRefView:
    # сторона пары адресуется числом: 2*строка — база, 2*строка+1 — кандидат.
    # плотную матрицу не разворачиваю, собираю на батч
    def __init__(self, embeddings, bidx, cidx, num_base, num_cand) -> None:
        self.embeddings = embeddings
        self.bidx = np.asarray(bidx)
        self.cidx = np.asarray(cidx)
        self.num_base = np.asarray(num_base, dtype=np.float32)
        self.num_cand = np.asarray(num_cand, dtype=np.float32)
        self.width = embeddings.shape[1] + self.num_base.shape[1]

    def take(self, refs: np.ndarray) -> np.ndarray:
        refs = np.asarray(refs)
        rows = refs >> 1
        is_cand = (refs & 1).astype(bool)
        emb_idx = np.where(is_cand, self.cidx[rows], self.bidx[rows])
        emb = self.embeddings[emb_idx].astype(np.float32, copy=False)
        num = np.where(is_cand[:, None], self.num_cand[rows], self.num_base[rows])
        return np.hstack([emb, num])


def _bucket_pick(pool: np.ndarray, codes: np.ndarray, queries: np.ndarray, rng):
    # случайный элемент пула с тем же кодом, иначе -1
    order = np.argsort(codes, kind="stable")
    sorted_codes = codes[order]
    lo = np.searchsorted(sorted_codes, queries, side="left")
    hi = np.searchsorted(sorted_codes, queries, side="right")
    size = hi - lo
    out = np.full(len(queries), -1, dtype=np.int64)
    ok = size > 0
    if ok.any():
        offset = (rng.random(ok.sum()) * size[ok]).astype(np.int64)
        out[ok] = pool[order[lo[ok] + offset]]
    return out


def mine_triplets(
    base_id: np.ndarray,
    y: np.ndarray,
    side_keys: np.ndarray,
    rows: np.ndarray,
    *,
    n_negatives: int = 1,
    seed: int = 42,
    verbose: bool = True,
):
    # side_keys — коды атрибутов формы [2*строк, 4]: param2, param1, подкатегория, категория
    if n_negatives > 1:
        parts, merged = [], {}
        for k in range(n_negatives):
            a, p, n, stats = _mine_once(
                base_id, y, side_keys, rows, seed=seed + k, verbose=False
            )
            parts.append((a, p, n))
            for name, count in stats.items():
                merged[name] = merged.get(name, 0) + count
        anchors = np.concatenate([x[0] for x in parts])
        positives = np.concatenate([x[1] for x in parts])
        negatives = np.concatenate([x[2] for x in parts])
        if verbose:
            _report(len(anchors), merged)
        return anchors, positives, negatives, merged
    return _mine_once(base_id, y, side_keys, rows, seed=seed, verbose=verbose)


def _report(total: int, stats: dict) -> None:
    print(f"триплетов собрано: {total}", flush=True)
    for name, count in stats.items():
        if count:
            print(f"  негатив с уровня «{name}»: {count} ({count / total:.1%})", flush=True)


def _mine_once(
    base_id: np.ndarray,
    y: np.ndarray,
    side_keys: np.ndarray,
    rows: np.ndarray,
    *,
    seed: int = 42,
    verbose: bool = True,
):
    rng = np.random.default_rng(seed)
    rows = np.asarray(rows)
    y_rows = y[rows]

    positive_rows = rows[y_rows == 1]
    negative_rows = rows[y_rows == 0]
    if len(positive_rows) == 0 or len(negative_rows) == 0:
        raise ValueError("для триплетов нужны и дубли, и не дубли")

    anchors = (positive_rows * 2).astype(np.int64)
    positives = anchors + 1
    negatives = np.full(len(anchors), -1, dtype=np.int64)

    # пул негативов — кандидаты из пар с меткой 0. они не дубли своей базы, и для чужого
    # якоря почти наверняка тоже
    pool = (negative_rows * 2 + 1).astype(np.int64)
    stats = {}

    # уровень 0: свои же кандидаты той же базы. самый тяжёлый негатив, какой есть в данных
    todo = negatives < 0
    picked = _bucket_pick(pool, base_id[negative_rows], base_id[positive_rows][todo], rng)
    negatives[todo] = picked
    stats[LEVELS[0]] = int((negatives >= 0).sum())

    # уровни 1..4: тот же param2, param1, подкатегория, категория
    for level in range(4):
        todo = negatives < 0
        if not todo.any():
            stats[LEVELS[level + 1]] = 0
            continue
        picked = _bucket_pick(
            pool, side_keys[pool, level], side_keys[anchors[todo], level], rng
        )
        negatives[todo] = picked
        stats[LEVELS[level + 1]] = int((negatives >= 0).sum() - sum(stats.values()))

    # остаток — случайный негатив
    todo = negatives < 0
    stats[LEVELS[5]] = int(todo.sum())
    if todo.any():
        negatives[todo] = rng.choice(pool, size=int(todo.sum()), replace=True)

    # негатив не может совпасть с якорем или позитивом
    bad = (negatives == anchors) | (negatives == positives)
    if bad.any():
        negatives[bad] = rng.choice(pool, size=int(bad.sum()), replace=True)

    if verbose:
        _report(len(anchors), stats)

    return anchors, positives, negatives, stats


class TripletTower:
    def __init__(
        self,
        input_dim: int,
        hidden: int = 256,
        output: int = 128,
        margin: float = DEFAULT_MARGIN,
        dropout: float = 0.1,
        lr: float = 1e-3,
        batch_size: int = 4096,
        epochs: int = 8,
        seed: int = 42,
        device: str | None = None,
        verbose: bool = True,
    ) -> None:
        self.input_dim = input_dim
        self.hidden = hidden
        self.output = output
        self.margin = margin
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.seed = seed
        self.verbose = verbose
        self.device = device or pick_device(verbose=verbose)
        self.model = None

    def fit(self, view: SideRefView, triplets) -> "TripletTower":
        import torch
        from torch.nn import TripletMarginLoss

        torch.manual_seed(self.seed)
        anchors, positives, negatives = triplets

        self.model = _build_module(self.input_dim, self.hidden, self.output, self.dropout)
        self.model = self.model.to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = TripletMarginLoss(margin=self.margin)

        rng = np.random.default_rng(self.seed)
        n = len(anchors)
        steps = max(n // self.batch_size, 1)

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            order = rng.permutation(n)
            total = 0.0
            for step in range(steps):
                rows = order[step * self.batch_size : (step + 1) * self.batch_size]
                a = torch.from_numpy(view.take(anchors[rows])).to(self.device)
                p = torch.from_numpy(view.take(positives[rows])).to(self.device)
                ng = torch.from_numpy(view.take(negatives[rows])).to(self.device)

                optimizer.zero_grad()
                loss = loss_fn(
                    self.model.encode(a), self.model.encode(p), self.model.encode(ng)
                )
                loss.backward()
                optimizer.step()
                total += loss.item()
            if self.verbose:
                print(f"  эпоха {epoch}: loss {total / steps:.4f}", flush=True)
        return self

    def predict(self, base_view, cand_view, batch_size: int = 16384) -> np.ndarray:
        # тот же скор, что у контрастной башни, чтобы его можно было подставить вместо неё
        import torch

        if self.model is None:
            raise RuntimeError("модель не обучена")
        self.model.eval()
        n = len(base_view)
        out = np.empty(n, dtype=np.float32)
        with torch.inference_mode():
            for start in range(0, n, batch_size):
                rows = np.arange(start, min(start + batch_size, n))
                b = torch.from_numpy(base_view.take(rows)).to(self.device)
                c = torch.from_numpy(cand_view.take(rows)).to(self.device)
                distance = self.model(b, c).cpu().numpy()
                out[rows] = 1.0 / (1.0 + distance)
        return out

## config

фотографии тут выключены: их кодирование занимает часы и покрывает данные частично,
на сравнение двух функций потерь это навесило бы шум

In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

MAX_ENTRIES_PER_DIR = 256
MAX_DIRS_VISITED = 400


def scan_dir(directory: Path, limit: int = MAX_ENTRIES_PER_DIR):
    files: list[Path] = []
    dirs: list[Path] = []
    try:
        with os.scandir(directory) as it:
            for i, entry in enumerate(it):
                if i >= limit:
                    break
                (dirs if entry.is_dir() else files).append(Path(entry.path))
    except OSError:
        pass
    return files, dirs


def walk_shallow(root: Path, max_depth: int = 4, skip=lambda p: False):
    queue: list[tuple[Path, int]] = [(root, 0)]
    visited = 0
    while queue and visited < MAX_DIRS_VISITED:
        directory, depth = queue.pop(0)
        if skip(directory):
            continue
        visited += 1
        files, dirs = scan_dir(directory)
        yield directory, files, dirs
        if depth < max_depth:
            queue.extend((d, depth + 1) for d in dirs)


def find_data_dir(root: Path = INPUT_ROOT) -> Path:
    seen: list[Path] = []
    for directory, files, _dirs in walk_shallow(
        root, skip=lambda p: "image" in p.name.lower()
    ):
        seen.append(directory)
        if any(f.name.startswith("train_part_") and f.suffix == ".parquet" for f in files):
            return directory
    listing = "\n".join(f"  {p}" for p in seen[:40])
    raise FileNotFoundError(
        f"под {root} нет файлов train_part_*.parquet.\n"
        f"Подключите датасет chuvirla/avito-ml-cup-default-dataset.\n"
        f"Просмотрены каталоги:\n{listing or '  (пусто)'}"
    )


DATA_DIR = find_data_dir()
print("данные:", DATA_DIR)

QUICK_RUN = False
N_SPLITS = 5
SEED = 42

TRIPLET_MARGIN = 0.3
NEGATIVES_PER_POSITIVE = 4
TRIPLET_EPOCHS = 12

TRAIN_FILES = sorted(DATA_DIR.glob("train_part_*.parquet"))
if QUICK_RUN:
    TRAIN_FILES = TRAIN_FILES[-1:]

print("train:", [f.name for f in TRAIN_FILES])
if not TRAIN_FILES:
    raise FileNotFoundError(f"в {DATA_DIR} нет обучающих частей")

## load and featurize

плюс коды атрибутов обеих сторон — по ним ищутся тяжёлые негативы

In [ ]:
FEATURE_COLUMNS_TEXT = [
    "base_title", "cand_title", "base_description", "cand_description",
    "base_category_name", "cand_category_name",
    "base_subcategory_name", "cand_subcategory_name",
    "base_param1", "cand_param1", "base_param2", "cand_param2",
    "base_json_params", "cand_json_params",
    "base_price", "cand_price", "base_count_images", "cand_count_images",
    "base_title_image", "cand_title_image",
    "is_same_location", "is_same_region",
    "base_item_id", "cand_item_id",
]

ATTRIBUTE_COLUMNS = ("category_name", "subcategory_name", "param1", "param2")


class Codes:
    # общий словарь «значение -> номер» для обеих сторон пары
    def __init__(self):
        self.map: dict = {}

    def encode(self, values) -> np.ndarray:
        m = self.map
        out = np.empty(len(values), dtype=np.int64)
        for i, v in enumerate(values):
            code = m.get(v)
            if code is None:
                code = len(m)
                m[v] = code
            out[i] = code
        return out


def attribute_codes(df, prefix, codes):
    return np.column_stack([
        codes[c].encode(df[f"{prefix}_{c}"].fillna("").astype(str).to_numpy())
        for c in ATTRIBUTE_COLUMNS
    ]).astype(np.int32)


def composite_keys(raw, codes):
    # четыре вложенных ключа: категория; +подкатегория; +param1; +param2. мощности словарей
    # берутся уже финальные, иначе один и тот же атрибут получил бы в разных файлах
    # разные коды
    cat, sub, p1, p2 = (raw[:, i].astype(np.int64) for i in range(4))
    k_cat = cat
    k_sub = k_cat * (len(codes["subcategory_name"].map) + 1) + sub
    k_p1 = k_sub * (len(codes["param1"].map) + 1) + p1
    k_p2 = k_p1 * (len(codes["param2"].map) + 1) + p2
    return np.column_stack([k_p2, k_p1, k_sub, k_cat])


def load_and_featurize(files, text_index):
    feats, labels, groups, base_ids, cand_ids, bidx, cidx = [], [], [], [], [], [], []
    keys_base, keys_cand = [], []
    codes = {c: Codes() for c in ATTRIBUTE_COLUMNS}

    for path in files:
        t0 = time.time()
        available = set(pq.ParquetFile(path).schema_arrow.names)
        cols = [c for c in FEATURE_COLUMNS_TEXT if c in available]
        cols += [c for c in ("is_double", "group_id") if c in available]

        df = pd.read_parquet(path, columns=cols)
        print(f"{path.name}: {len(df)} пар", flush=True)

        feats.append(build_features(df, verbose=True))
        base_ids.append(df["base_item_id"].to_numpy())
        cand_ids.append(df["cand_item_id"].to_numpy())
        labels.append(df["is_double"].to_numpy())
        groups.append(df["group_id"].to_numpy())

        b_texts = [
            build_bert_text(t, d)
            for t, d in zip(df["base_title"].fillna(""), df["base_description"].fillna(""))
        ]
        c_texts = [
            build_bert_text(t, d)
            for t, d in zip(df["cand_title"].fillna(""), df["cand_description"].fillna(""))
        ]
        bidx.append(text_index.add_many(b_texts))
        cidx.append(text_index.add_many(c_texts))
        del b_texts, c_texts

        keys_base.append(attribute_codes(df, "base", codes))
        keys_cand.append(attribute_codes(df, "cand", codes))

        del df
        gc.collect()
        print(f"  готово за {time.time() - t0:.0f}s", flush=True)

    out = {
        "X": pd.concat(feats, ignore_index=True),
        "base_id": np.concatenate(base_ids),
        "cand_id": np.concatenate(cand_ids),
        "y": np.concatenate(labels),
        "group": np.concatenate(groups),
        "bidx": np.concatenate(bidx),
        "cidx": np.concatenate(cidx),
    }
    kb = composite_keys(np.vstack(keys_base), codes)
    kc = composite_keys(np.vstack(keys_cand), codes)
    side_keys = np.empty((2 * len(kb), 4), dtype=np.int64)
    side_keys[0::2] = kb
    side_keys[1::2] = kc
    out["side_keys"] = side_keys
    return out


text_index = TextIndex()
t0 = time.time()
train = load_and_featurize(TRAIN_FILES, text_index)
print(f"\nвсего признаков собрано за {time.time() - t0:.0f}s")
print("train:", train["X"].shape, "| доля дублей: %.4f" % train["y"].mean())
print("уникальных текстов:", len(text_index))

base_code = pd.factorize(train["base_id"])[0].astype(np.int64)

## embedding titles

In [ ]:
t0 = time.time()
embeddings = encode_texts(text_index.texts, batch_size=512, max_length=64)
print(f"эмбеддинги {embeddings.shape} за {time.time() - t0:.0f}s")

train["X"] = pd.concat(
    [train["X"], embedding_pair_features(embeddings, train["bidx"], train["cidx"])],
    axis=1,
)
print("признаков после эмбеддингов:", train["X"].shape[1])

## side features for towers

In [ ]:
def side_numeric(files, prefix):
    parts = []
    for path in files:
        df = pd.read_parquet(
            path,
            columns=[f"{prefix}_price", f"{prefix}_count_images",
                     f"{prefix}_title", f"{prefix}_description"],
        )
        parts.append(side_numeric_features(df, prefix))
        del df
    return np.vstack(parts)


num_base = side_numeric(TRAIN_FILES, "base")
num_cand = side_numeric(TRAIN_FILES, "cand")
stats_source = np.vstack([num_base, num_cand])
mean, std = stats_source.mean(0), stats_source.std(0) + 1e-6
del stats_source
gc.collect()

std_base = ((num_base - mean) / std).astype(np.float32)
std_cand = ((num_cand - mean) / std).astype(np.float32)

tower_base = SideView(embeddings, train["bidx"], std_base)
tower_cand = SideView(embeddings, train["cidx"], std_cand)
ref_view = SideRefView(embeddings, train["bidx"], train["cidx"], std_base, std_cand)
TOWER_WIDTH = tower_base.shape[1]
print("вход башни:", tower_base.shape)

## hard negative mining

негатив ищу от узкого бакета к широкому: свои же кандидаты той же базы с меткой 0,
потом тот же `param2`, `param1`, подкатегория, категория. случайный негатив из чужой
категории отделяется тривиально, и башня упирается в потолок

In [ ]:
folds = group_folds(train["group"], n_splits=N_SPLITS, seed=SEED)

_, _, _, mining_stats = mine_triplets(
    base_code, train["y"], train["side_keys"], np.arange(len(train["y"])),
    n_negatives=1, seed=SEED,
)

## towers

энкодер у обеих один и тот же, отличается только ошибка. контрастная тянет абсолютные
расстояния: дубли к нулю, недубли за зазор. триплетная оптимизирует порядок — позитив
ближе негатива хотя бы на зазор, а MAP спрашивает ровно про порядок. скор в обоих
случаях `1 / (1 + d)`, так что бустинг разницы между схемами не видит

In [ ]:
TOWER_KWARGS = dict(hidden=256, output=128, margin=1.0, epochs=8, batch_size=4096, seed=SEED)
TRIPLET_KWARGS = dict(
    hidden=256, output=128, margin=TRIPLET_MARGIN,
    epochs=TRIPLET_EPOCHS, batch_size=4096, seed=SEED,
)
INNER_SPLITS = 4


def fit_tower(kind, rows):
    if kind == "triplet":
        anchors, positives, negatives, _ = mine_triplets(
            base_code, train["y"], train["side_keys"], rows,
            n_negatives=NEGATIVES_PER_POSITIVE, seed=SEED, verbose=False,
        )
        model = TripletTower(input_dim=TOWER_WIDTH, verbose=False, **TRIPLET_KWARGS)
        return model.fit(ref_view, (anchors, positives, negatives))
    model = TwoTowerContrastive(input_dim=TOWER_WIDTH, verbose=False, **TOWER_KWARGS)
    return model.fit(
        tower_base.subset(rows), tower_cand.subset(rows), train["y"][rows]
    )


def tower_scores_for_fold(kind, tr, va):
    sim_tr = np.zeros(len(tr), dtype=np.float32)
    for inner_tr, inner_va in group_folds(train["group"][tr], INNER_SPLITS, seed=SEED + 1):
        model = fit_tower(kind, tr[inner_tr])
        sim_tr[inner_va] = model.predict(
            tower_base.subset(tr[inner_va]), tower_cand.subset(tr[inner_va])
        )
        del model
        gc.collect()
    full = fit_tower(kind, tr)
    sim_va = full.predict(tower_base.subset(va), tower_cand.subset(va))
    del full
    gc.collect()
    return sim_tr, sim_va

## tower alone

как каждая схема ранжирует пары сама по себе, до бустинга. фолд 1

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

solo = {}
tr0, va0 = folds[0]
for kind in ("contrastive", "triplet"):
    t0 = time.time()
    model = fit_tower(kind, tr0)
    score = model.predict(tower_base.subset(va0), tower_cand.subset(va0))
    solo[kind] = {
        "pr_auc": float(average_precision_score(train["y"][va0], score)),
        "roc_auc": float(roc_auc_score(train["y"][va0], score)),
        "map": float(mean_average_precision(train["base_id"][va0], train["y"][va0], score)),
        "seconds": time.time() - t0,
    }
    print(f"{kind:12} PR-AUC {solo[kind]['pr_auc']:.4f} | "
          f"ROC-AUC {solo[kind]['roc_auc']:.4f} | MAP {solo[kind]['map']:.4f} | "
          f"{solo[kind]['seconds']:.0f}s", flush=True)
    del model
    gc.collect()

## training

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(
    objective="binary",
    metric="average_precision",
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    n_estimators=1200,
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
)


def run_cv(kind, label):
    y, base_ids = train["y"], train["base_id"]
    oof = np.zeros(len(y))
    rows_report = []
    models = []

    for i, (tr, va) in enumerate(folds):
        X_tr, X_va = train["X"].iloc[tr], train["X"].iloc[va]
        if kind is not None:
            sim_tr, sim_va = tower_scores_for_fold(kind, tr, va)
            X_tr, X_va = X_tr.copy(), X_va.copy()
            X_tr["tower_sim"], X_va["tower_sim"] = sim_tr, sim_va

        pos_weight = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
        model = lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=pos_weight)
        model.fit(
            X_tr, y[tr],
            eval_set=[(X_va, y[va])],
            callbacks=[
                lgb.early_stopping(100, first_metric_only=True, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        p = model.predict_proba(X_va)[:, 1]
        oof[va] = p
        models.append(model)
        rows_report.append({
            "fold": i + 1,
            "pr_auc": average_precision_score(y[va], p),
            "roc_auc": roc_auc_score(y[va], p),
            "map": mean_average_precision(base_ids[va], y[va], p),
        })
        print(f"  фолд {i + 1}: PR-AUC {rows_report[-1]['pr_auc']:.4f}", flush=True)
        del X_tr, X_va
        gc.collect()

    result = {
        "pr_auc": float(average_precision_score(y, oof)),
        "roc_auc": float(roc_auc_score(y, oof)),
        "map": float(mean_average_precision(base_ids, y, oof)),
        "per_fold": rows_report,
    }
    print(f"{label}: OOF PR-AUC {result['pr_auc']:.4f} | "
          f"ROC-AUC {result['roc_auc']:.4f} | MAP {result['map']:.4f}\n", flush=True)
    return result, oof, models


cv = {}
print("без башни:")
cv["без башни"], oof_plain, _ = run_cv(None, "без башни")
print("контрастная башня:")
cv["контрастная"], oof_contrastive, _ = run_cv("contrastive", "контрастная")
print("триплетная башня:")
cv["триплетная"], oof_triplet, models_triplet = run_cv("triplet", "триплетная")

## summary

In [ ]:
print(f"{'схема':<16} {'PR-AUC':>8} {'ROC-AUC':>9} {'MAP':>8}")
for name, r in cv.items():
    print(f"{name:<16} {r['pr_auc']:>8.4f} {r['roc_auc']:>9.4f} {r['map']:>8.4f}")

delta = cv["триплетная"]["pr_auc"] - cv["контрастная"]["pr_auc"]
print(f"\nтриплеты против контрастной: {delta:+.4f} PR-AUC")

importance = pd.DataFrame({
    "feature": models_triplet[0].booster_.feature_name(),
    "gain": np.mean([m.booster_.feature_importance("gain") for m in models_triplet], axis=0),
}).sort_values("gain", ascending=False).reset_index(drop=True)
print()
print(importance.head(15).to_string(index=False))

results = {
    "n_train_pairs": int(len(train["y"])),
    "positive_rate": float(train["y"].mean()),
    "n_features": int(train["X"].shape[1]),
    "triplet": {
        "margin": TRIPLET_MARGIN,
        "negatives_per_positive": NEGATIVES_PER_POSITIVE,
        "epochs": TRIPLET_EPOCHS,
        "mining_levels": mining_stats,
    },
    "tower_alone_fold1": solo,
    "cv": cv,
    "top_features": importance.head(20).to_dict("records"),
}
np.savez_compressed(
    OUTPUT_DIR / "oof_triplet.npz",
    y=train["y"], base_id=train["base_id"],
    plain=oof_plain, contrastive=oof_contrastive, triplet=oof_triplet,
)
with open(OUTPUT_DIR / "results_triplet.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=float)
print()
print(json.dumps(cv, ensure_ascii=False, indent=2, default=float))